# Tech Challenge - Fase 1: Análise de Dados e Modelagem Preditiva

Este notebook contém a exploração de dados, o pré-processamento e a criação de modelos preditivos para classificar a presença de diabetes com base em dados tabulares clínicos. 

O objetivo principal é construir uma ferramenta de suporte à decisão médica para otimizar o tempo de triagem. Lembre-se: o modelo serve como auxílio diagnóstico, e a palavra final sempre cabe ao profissional de saúde.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

## Exploração de Dados

Vamos carregar o arquivo de dados e examinar suas propriedades estruturais, tipos de variáveis e estatísticas descritivas básicas.

In [ ]:
# Carregando a base de dados
df = pd.read_csv('diabetes.csv')

# Exibindo as primeiras linhas do conjunto de dados
print("Primeiras linhas do dataset:")
display(df.head())

# Informações sobre os tipos de dados e valores nulos aparentes
print("\nInformações gerais do dataset:")
display(df.info())

# Estatísticas descritivas das variáveis numéricas
print("\nEstatísticas descritivas:")
display(df.describe())

# Visualização da distribuição da variável alvo
plt.figure(figsize=(6, 4))
sns.countplot(x='Outcome', data=df, palette='viridis')
plt.title('Distribuição do Diagnóstico (0 = Negativo, 1 = Positivo)')
plt.xlabel('Resultado (Outcome)')
plt.ylabel('Quantidade de Pacientes')
plt.show()

### Análise de Correlação

Análise visual para identificar o nível de associação linear entre os atributos clínicos e a variável de diagnóstico.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Matriz de Correlação entre Variáveis')
plt.show()

## Pré-processamento de Dados

Considerando que valores iguais a zero em atributos como Glicose, Pressão Arterial e IMC frequentemente indicam ausência de registro ou erro de preenchimento. Vamos converter esses zeros inválidos em valores nulos e tratá-los utilizando um pipeline com substituição pela mediana e padronização estatística.

In [ ]:
# Substituindo zeros inconsistentes por NaN
colunas_com_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[colunas_com_zeros] = df[colunas_com_zeros].replace(0, np.nan)

# Separação de features (X) e rótulo (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Separação em conjuntos de treino e teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Construção do pipeline de pré-processamento em Python
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Ajuste e transformação dos dados
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

## Modelagem e Treinamento

Nesta etapa, vamos instanciar os algoritmos de classificação escolhidos e treiná-los utilizando o conjunto de dados de treino preparado anteriormente. Utilizaremos a Regressão Logística e o Random Forest para fins de comparação.

In [ ]:
# Instanciando e treinando o modelo de Regressão Logística
modelo_lr = LogisticRegression(random_state=42)
modelo_lr.fit(X_train_processed, y_train)

# Instanciando e treinando o modelo de Random Forest
modelo_rf = RandomForestClassifier(random_state=42, n_estimators=100)
modelo_rf.fit(X_train_processed, y_train)

## Avaliação dos Modelos

Com os modelos já treinados, vamos avaliar o desempenho de cada um utilizando os dados de teste (que foram separados e não foram vistos durante o treinamento). Analisaremos métricas como Acurácia, Recall e F1-Score, além da Matriz de Confusão, para entender o comportamento de cada solução.

In [ ]:
def avaliar_modelo(nome, modelo, X_dados, y_reais):
    predicoes = modelo.predict(X_dados)
    
    acuracia = accuracy_score(y_reais, predicoes)
    recall = recall_score(y_reais, predicoes)
    f1 = f1_score(y_reais, predicoes)
    
    print(f"--- Desempenho do Modelo: {nome} ---")
    print(f"Acurácia:  {acuracia:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")
    
    # Matriz de Confusão
    plt.figure(figsize=(5, 4))
    sns.heatmap(confusion_matrix(y_reais, predicoes), annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusão - {nome}')
    plt.xlabel('Classificação do Modelo')
    plt.ylabel('Diagnóstico Real')
    plt.show()

# Executando a avaliação para ambos os modelos
avaliar_modelo("Regressão Logística", modelo_lr, X_test_processed, y_test)
avaliar_modelo("Random Forest", modelo_rf, X_test_processed, y_test)

## Interpretabilidade dos Resultados

Para garantir a transparência do sistema preditivo, utilizamos técnicas de importância de atributos do próprio modelo e a biblioteca SHAP para explicar a contribuição de cada variável no diagnóstico final.

In [ ]:
# Importância dos Atributos - Random Forest
importances = modelo_rf.feature_importances_
indices = np.argsort(importances)[::-1]
nomes_atributos = X.columns

plt.figure(figsize=(10, 6))
plt.title("Importância das Variáveis no Modelo Random Forest")
plt.bar(range(X.shape[1]), importances[indices], align="center", color='skyblue')
plt.xticks(range(X.shape[1]), nomes_atributos[indices], rotation=45)
plt.ylabel("Peso Relativo")
plt.tight_layout()
plt.show()

# Explicação com valores SHAP
explainer = shap.TreeExplainer(modelo_rf)
shap_values = explainer.shap_values(X_test_processed)

# Exibindo o gráfico de resumo para a classe positiva (índice 1)
shap.summary_plot(shap_values[:, :, 1], X_test_processed, feature_names=nomes_atributos)

## Conclusão Crítica

A análise de importância de variáveis e o SHAP demonstram que os níveis de Glicose e o Índice de Massa Corporal (IMC) exercem a maior influência na decisão dos modelos.

**O modelo pode ser utilizado na prática clínica?**
Sim, contanto que seja integrado estritamente como um mecanismo de **triagem inicial**. O sistema é capaz de filtrar exames de alta probabilidade de risco para que recebam atenção prioritária nas filas de atendimento. Devido à presença de margens de erro inerentes a qualquer algoritmo estatístico, as previsões geradas nunca devem substituir a validação laboratorial tradicional ou o julgamento clínico humano; o médico detém sempre a decisão final no diagnóstico.